# SMELL EXPERIMENT TUTORIAL

# Objectives

This tutorial demonstrates how Interactional Motivation (IM) affects the agent's behavior. 
IM is used to implement an "instinctive behavior" to approach targets that "smell good". 
The agent initially ignores the meaning of possible actions and sensory signals. 
In short, it does not know what it is doing but it knows whether it likes it or not.

We expect the agent to learn patterns of behaviors that allow it to ascend the gradient of smell.
Depending on the intial motivational settings, we will observe different kinds of behaviors, which demonstrates the learning.

The key difference with traditional reinforcement-learning algorithms is that the reward is not associated with the state of reaching the target (traditional extrinsic reward). 
Instead, IM drives a schema mechanism that works in non-Markovian settings, which allwos its easy transfer to robots in the open world.  

See the full tutorial on Open World Schema Mechanism driven by Interactional Motivation at https://github.com/PetiteIA/schema_mechanism

# Let's implement the agent

In [1]:
class CompositeInteraction:
    """A composite interaction is a tuple (pre_interaction, post_interaction) and a weight"""
    def __init__(self, pre_interaction, post_interaction):
        self.pre_interaction = pre_interaction
        self.post_interaction = post_interaction
        self.weight = 1
        self._step = 1

    def get_decision(self):
        """Return the flatten sequence of intermediary primitive interactions terminated with the final decision"""
        return f"{self.pre_interaction.sequence()}{self.post_interaction.get_decision()}"

    def get_actions(self):
        """Return the flat sequence of the decisions of this interaction as a string"""
        return f"{self.pre_interaction.get_actions()}{self.post_interaction.get_actions()}"

    def get_valence(self):
        """Return the valence of the pre_interaction plus the valence of the post_interaction"""
        return self.pre_interaction.get_valence() + self.post_interaction.get_valence()

    def reinforce(self):
        """Increment the composite interaction's weight"""
        self.weight += 1

    def key(self):
        """ The key to find this interaction in the dictionary is the string '<pre_interaction><post_interaction>'. """
        return f"({self.pre_interaction.key()},{self.post_interaction.key()})"

    def pre_key(self):
        """Return the key of the pre_interaction"""
        #if self.weight > confidence_threshold:
        return self.pre_interaction.key()
        #else:
        #return self.pre_interaction.pre_key()

    def __str__(self):
        """ Print the interaction in the Newick tree format (pre_interaction, post_interaction: valence) """
        return f"({self.pre_interaction}, {self.post_interaction}: {self.weight})"

    def __eq__(self, other):
        """ Interactions are equal if they have the same pre and post interactions """
        if isinstance(other, self.__class__):
            return (self.pre_interaction == other.pre_interaction) and (self.post_interaction == other.post_interaction)
        else:
            return False

    def get_length(self):
        """Return the length of the number of primitive interactions in this composite interaction"""
        return self.pre_interaction.get_length() + self.post_interaction.get_length()

    def increment(self, interaction, interactions):
        """Increment the step of the appropriate sub-interaction. Return the enacted interaction if it is over, or None if it is ongoing."""
        # First step 
        if self._step == 1:
            interaction = self.pre_interaction.increment(interaction, interactions)
            # Ongoing pre-interaction. Return None
            if interaction is None:
                return None
            # Pre-interaction succeeded. Increment the step and return None
            elif interaction == self.pre_interaction:
                self._step = 2
                return None
            # Pre-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                return interaction
        # Second step
        else:
            interaction = self.post_interaction.increment(interaction, interactions)
            # Ongoing post-interaction. Return None
            if interaction is None:
                return None
            # Post-interaction succeeded. Reset the step and return this interaction
            elif interaction == self.post_interaction:
                self._step = 1
                return self
            # Post-interaction failed. Reset the step and return the enacted interaction
            else:
                self._step = 1
                composite_interaction = CompositeInteraction(self.pre_interaction, interaction)
                if composite_interaction.key() not in interactions:
                    # Add the enacted composite interaction to memory
                    interactions[composite_interaction.key()] = composite_interaction
                    if trace:
                        print(f"Learning {composite_interaction}")
                    return composite_interaction
                else:
                    # Reinforce the existing composite interaction and return it
                    interactions[composite_interaction.key()].reinforce()
                    if trace:
                        print(f"Reinforcing {interactions[composite_interaction.key()]}")
                    return interactions[composite_interaction.key()]

    def current(self):
        """Return the current intended primitive interaction"""
        # Step 1: the current primitive interaction of the pre-interaction
        if self._step == 1:
            return self.pre_interaction.current()
        # Step 2: The current primitive interaction of the post-interaction
        else:
            return self.post_interaction.current()

    def sequence(self):
        """Return the flat sequence of primitive interactions of this composite interaction"""
        return f"{self.pre_interaction.sequence()}{self.post_interaction.sequence()}"

    def get_post_interactions(self):
        """Return the list of the hierarchy of the sub post_interactions"""
        return [self.post_interaction] + self.post_interaction.get_post_interactions()

In [2]:
class Interaction:
    """An interaction is a tuple (action, outcome) with a valence"""
    def __init__(self, _action, _outcome, _valence):
        self._action = _action
        self._outcome = _outcome
        self._valence = _valence
        self.weight = 10
        
    def get_action(self):
        """Return the action"""
        return self._action

    def get_actions(self):
        """Return the action as a string for compatibilty with CompositeInteraction"""
        return str(self._action)

    def get_decision(self):
        """Return the decision key"""
        return f"{self._action}"
        # return f"a{self._action}"

    def get_outcome(self):
        """Return the action"""
        return self._outcome

    def get_valence(self):
        """Return the action"""
        return self._valence

    def key(self):
        """ The key to find this interaction in the dictinary is the string '<action><outcome>'. """
        return f"{self._action}{self._outcome}"

    def pre_key(self):
        """Return the key. Used for compatibility with CompositeInteraction"""
        return ""  # self.key()

    def __str__(self):
        """ Print interaction in the form '<action><outcome:<valence>' for debug."""
        return f"{self._action}{self._outcome}:{self._valence}"

    def __eq__(self, other):
        """ Interactions are equal if they have the same key """
        if isinstance(other, self.__class__):
            return self.key() == other.key()
        else:
            return False

    def get_length(self):
        """The length of the sequence of this interaction"""
        return 1

    def increment(self, interaction, interactions):
        """Return the enacted interaction for compatibility with composite interactions"""
        return interaction

    def current(self):
        """Return itself for compatibility with composite interactions"""
        return self

    def sequence(self):
        """Return the key. Use for compatibility with composite interactions"""
        return self.key()

    def get_post_interactions(self):
        """Return the empty list for compatibility with composite interactions"""
        return []

In [3]:
trace = True
# Maximum length of intended sequence
max_length = 5
# Minimum weight of intended sequence
min_weight = 3

In [4]:
import pandas as pd

class Agent:
    def __init__(self, _interactions):
        """ Initialize our agent """
        self._interactions = {interaction.key(): interaction for interaction in _interactions}
        self._primitive_intended_interaction = self._interactions["00"]
        # self._primitive_intended_interaction = self._interactions[f"{SNIFF_FRONT}{STABLE}"]
        self._intended_interaction = None

        # The context
        self._penultimate_interaction = None
        self._previous_interaction = None
        self._last_interaction = None
        self._penultimate_composite_interaction = None
        self._previous_composite_interaction = None
        self._last_composite_interaction = None
        
        # Prepare the dataframe of proposed interactions
        default_interactions = [interaction for interaction in _interactions if interaction.get_outcome() == 0]
        data = {'activated': [""] * len(default_interactions),
                'weight': [0] * len(default_interactions),
                'actions': [i.get_actions() for i in default_interactions],
                'intention': [i.key() for i in default_interactions],
                'valence': [i.get_valence() for i in default_interactions],
                'decision': [i.get_decision() for i in default_interactions],
                'length': [1] * len(default_interactions),
                'pre': [""] * len(default_interactions)} 
        self._default_df = pd.DataFrame(data)
        self.proposed_df = None
        self.decision_df = None
        self.clear = True # Used to clear the display after the enacted interaction

    def action(self, _outcome):
        """Implement the agent's policy"""
        # Trace the previous cycle
        primitive_enacted_interaction = self._interactions[f"{self._primitive_intended_interaction.get_action()}{_outcome}"]
        if trace:
            print(
            f"Action: {self._primitive_intended_interaction.get_action()}, Prediction: {self._primitive_intended_interaction.get_outcome()}, "
            f"Outcome: {_outcome}, Prediction_correct: {self._primitive_intended_interaction.get_outcome() == _outcome}, "
            f"Valence: {primitive_enacted_interaction.get_valence()}")

        # Follow up the enaction
        if self._intended_interaction is None: # First interaction cycle
            enacted_interaction = primitive_enacted_interaction
        else:
            enacted_interaction = self._intended_interaction.increment(primitive_enacted_interaction, self._interactions)

        # If the intended interaction is over (completely enacted or aborted)
        if enacted_interaction is None:
            self.clear = False
        else:
            self.clear = True
            # Memorize the context
            self._penultimate_composite_interaction = self._previous_composite_interaction
            self._previous_composite_interaction = self._last_composite_interaction
            self._penultimate_interaction = self._previous_interaction
            self._previous_interaction = self._last_interaction
            self._last_interaction = enacted_interaction
            # Call the learning mechanism
            self.learn(enacted_interaction)
            # Create the proposed dataframe
            self.create_proposed_df()
            self.aggregate_propositions()
            # Decide the next enaction
            self.decide()

        # Return the next primitive action
        self._primitive_intended_interaction = self._intended_interaction.current()
        return self._primitive_intended_interaction.get_action()
        
    def learn(self, enacted_interaction):
        """Learn the composite interactions"""
        # First level of composite interactions
        self._last_composite_interaction = self.learn_composite_interaction(self._previous_interaction, enacted_interaction)
        # Second level of composite interactions
        self.learn_composite_interaction(self._previous_composite_interaction, enacted_interaction)
        self.learn_composite_interaction(self._penultimate_interaction, self._last_composite_interaction)

        # Higher level composite interaction made of two composite interactions
        if self._last_composite_interaction is not None:
            self.learn_composite_interaction(self._penultimate_composite_interaction, self._last_composite_interaction)

    def learn_composite_interaction(self, pre_interaction, post_interaction):
        """Record or reinforce the composite interaction made of (pre_interaction, post_interaction)"""
        if pre_interaction is None:
            return None
        else:
            # If the pre-interaction exists
            composite_interaction = CompositeInteraction(pre_interaction, post_interaction)
            if composite_interaction.key() not in self._interactions:
                # Add the composite interaction to memory
                self._interactions[composite_interaction.key()] = composite_interaction
                if trace:
                    print(f"Learning {composite_interaction}")
                return composite_interaction
            else:
                # Reinforce the existing composite interaction and return it
                self._interactions[composite_interaction.key()].reinforce()
                if trace:
                    print(f"Reinforcing {self._interactions[composite_interaction.key()]}")
                return self._interactions[composite_interaction.key()]

    def create_proposed_df(self):
        """Create the proposed dataframe from the activated interactions"""
        # The list of activated interactions that match the current context
        activated_interactions = [i for i in self._interactions.values() if i.get_length() > 1 
                                  and i.pre_interaction in self._last_composite_interaction.get_post_interactions()]
        data = {'activated': [i.key() for i in activated_interactions],
                'weight': [i.weight for i in activated_interactions],
                'actions': [i.post_interaction.get_actions() for i in activated_interactions],
                'intention': [i.post_interaction.key() for i in activated_interactions],
                'valence': [i.post_interaction.get_valence() for i in activated_interactions],
                'decision': [i.post_interaction.get_decision() for i in activated_interactions],
                'pre': [i.post_interaction.pre_key() for i in activated_interactions],
                'length': [i.post_interaction.get_length() for i in activated_interactions],
                }
        activated_df = pd.DataFrame(data).astype(self._default_df.dtypes)  # Force the same types in case activated_df is empty

        # Create the proposed dataframe
        self.proposed_df = pd.concat([self._default_df, activated_df], ignore_index=True).sort_values(by='decision', ascending=True).reset_index(drop=True)

        # Calculate the proclivity of each proposition
        self.proposed_df['proclivity'] = self.proposed_df['weight'] * self.proposed_df['valence']

        # Compute the probability of each propositions
        # self.proposed_df['probability'] = self.proposed_df['weight'] / self.proposed_df.groupby('actions')['weight'].transform('sum')
        # self.proposed_df['probability'] = self.proposed_df.groupby('intention')['weight'].transform('sum') / self.proposed_df.groupby('actions')['weight'].transform('sum')

    def aggregate_propositions(self):
        """Aggregate the proclivity"""
        # Aggregate the proclivity for each decision
        grouped_df = self.proposed_df.groupby('decision').agg({'proclivity': 'sum', 'actions': 'first', # 'action': 'first', 
                                                               'length': 'first', 'intention': 'first', 'pre': 'first'}).reset_index()
        # For each proposed composite decision 
        for index, proposed in grouped_df[grouped_df['length'] > 1].iterrows():
            # print(f"Index {index}, actions {proposition['actions']}, intention {proposition['intention']}")
            # Find shorter decisions that start with the same sequence 
            for _, shorter in self.proposed_df[self.proposed_df.apply(lambda row: proposed['actions'].startswith(row['actions']) 
                                                                      and row['length'] < proposed['length'], axis=1)].iterrows():
                # Add the proclivity of the shorter decisions
                grouped_df.loc[index, 'proclivity'] += shorter['proclivity']
                # print(f"Decision {proposed['decision']} recieves {shorter['proclivity']} from shorter {shorter['intention']}")

        # Remove the intentions that are insufficiently reinforced
        grouped_df = grouped_df[grouped_df['intention'].apply(lambda x: self._interactions[x].weight) >= min_weight]
        # Remove the intentions that are too long
        grouped_df = grouped_df[grouped_df['intention'].apply(lambda x: self._interactions[x].get_length()) <= max_length]
        
        # Sort by descending proclivity
        self.decision_df = grouped_df.sort_values(by=['proclivity', 'decision'], ascending=[False, True]).reset_index(drop=True)

    def decide(self):
        """Selects the intended_interaction at the top of the proposed dataframe"""
        # The intended interaction is in the first row because it has been sorted by descending proclivity
        intended_interaction_key = self.decision_df.loc[0, 'intention']
        if trace:
            print("Intention:", intended_interaction_key)
        self._intended_interaction = self._interactions[intended_interaction_key]

# Let's implement the environment 

## Define the possibilities of interaction

This environment can execute three actions: 
* `move_forward`
* `turn_left`
* `turn_right` 

which may yield six possible outcomes depending on the smell in the cell in front of the agent:

* `decrease`: the smell decreased
* `stable`: the smell remained stable
* `increase`: the smell increased
* `bump` : the agent collided with a wall (only after `move_forward`)
* `eat`: the agent reached the target and ate it

In [5]:
# Actions
FORWARD = 0
TURN_LEFT = 1
TURN_RIGHT = 2

# Outcomes
DECREASE = 0
STABLE = 1
INCREASE = 2
BUMP = 3
EAT = 4

## Define the grid

In [6]:
import numpy as np

# 0: Empty cell. 1: Wall
grid = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
])

## Define the environment class

In [7]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_hex, to_rgb
from ipywidgets import Button, HBox, VBox, Output
from IPython.display import display

BUMPING = 4
TARGET = 5
agent_color = "#1976D2"
grid_colors = ["#D6D6D6", '#5C946E', '#FAE2DB', '#535865', "#F93943", "#E365C1"]
# Directions
LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3
MAX_SMELL = 100

class Environment():
    def __init__(self, position, direction):
        self.grid = grid.copy()
        self.maze = grid.copy()
        self.position = np.array(position)  # Using NumPy array of shape (2)
        self.direction = direction
        self.colors = np.array([to_rgb(c) for c in grid_colors])
        brighter_target =  to_hex(self.colors[5] + (1 - self.colors[5]) * 0.2)
        self.cmap_smell = LinearSegmentedColormap.from_list("smell_gradiant", [brighter_target, "#ffffff"])
        self.marker_size = 400
        self.marker_map = {LEFT: '<', DOWN: 'v', RIGHT: '>', UP: '^'}
        self.marker_color = agent_color
        self.directions = np.array([
            [0, -1],  # Left
            [1, 0],   # Down
            [0, 1],   # Right
            [-1, 0]   # Up
        ])
        self.smell_grid = np.full(self.grid.shape, MAX_SMELL)
        self.fig = None  # , _ = plt.subplots()    
        # --- Build the smell graph ---
        self.G = nx.Graph()
        # Connect all adjacent (non-wall) cells
        for r in range(self.grid.shape[0]):
            for c in range(self.grid.shape[1]):
                if self.grid[r, c] == 1:
                    continue  # skip walls
                for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < self.grid.shape[0] and 0 <= nc < self.grid.shape[1] and self.grid[nr, nc] != 1:
                        self.G.add_edge((r, c), (nr, nc), weight=1)

    def outcome(self, action):
        """Update the grid. Return the outcome of the action."""
        result = STABLE

        front_position = self.position + self.directions[self.direction]
        left_position = self.position + self.directions[(self.direction + 1) % 4]
        right_position = self.position + self.directions[self.direction - 1]
        position_smell = self.smell_grid[tuple(self.position)]
        front_smell = self.smell_grid[tuple(front_position)]
        left_smell = self.smell_grid[tuple(left_position)]
        right_smell = self.smell_grid[tuple(right_position)]

        if action == FORWARD:  
            if self.grid[tuple(front_position)] in [0, TARGET]:  # Don't bump in targets
                self.position[:] = front_position
                new_front_smell = self.smell_grid[tuple(self.position + self.directions[self.direction])]
                result = STABLE + (new_front_smell < front_smell) - (new_front_smell > front_smell)
            else:
                result = BUMP
                self.maze[tuple(front_position)] = BUMPING
        
        elif action == TURN_RIGHT:
            result = STABLE + (right_smell < front_smell) - (right_smell > front_smell)
            self.direction = (self.direction + 3) % 4
        
        elif action == TURN_LEFT:
            result = STABLE + (left_smell < front_smell) - (left_smell > front_smell)
            self.direction = (self.direction + 1) % 4
        
        elif action == FEEL_FRONT:
            if self.grid[tuple(front_position)] == 0:
                self.maze[tuple(front_position)] = FEELING_EMPTY
                result = STABLE + (front_smell < position_smell) - (front_smell > position_smell)
            else:
                result = BUMP
                self.maze[tuple(front_position)] = FEELING_WALL
        
        elif action == FEEL_LEFT:
            if self.grid[tuple(left_position)] == 0:
                result = STABLE + (left_smell < position_smell) - (left_smell > position_smell)
                self.maze[tuple(left_position)] = FEELING_EMPTY
            else:
                result = BUMP
                self.maze[tuple(left_position)] = FEELING_WALL
        
        elif action == FEEL_RIGHT:
            if self.grid[tuple(right_position)] == 0:
                result = STABLE + (right_smell < position_smell) - (right_smell > position_smell)
                self.maze[tuple(right_position)] = FEELING_EMPTY
            else:
                result = BUMP
                self.maze[tuple(right_position)] = FEELING_WALL

        # Eat
        if self.grid[tuple(self.position)] == TARGET:
            self.clear_target(tuple(self.position))
            result = EAT
        
        # print(f"Line: {self.position[0]}, Column: {self.position[1]}, direction: {self.direction}")
        return result  
    
    def display(self):
        """Display the grid in the notebook"""
        out.clear_output(wait=True)
        with out:
            # ChatGPT recommends closing and recreating the figure
            plt.close(self.fig)
            self.fig, ax = plt.subplots()
            ax.imshow(self.colors[self.maze])
            plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
            masked_smell = np.ma.masked_where(self.grid > 0, self.smell_grid)
            ax.imshow(masked_smell, cmap=self.cmap_smell, vmin=0, vmax=np.max(masked_smell))
            ax.text(self.grid.shape[1]-1.5, 0.2, f"{step:>3}", fontsize=12, color='White')
            plt.show()
            cid = self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        
    def on_click(self, event):
        """Add or remove a target when the user clicks on the grid in widget mode"""
        if event.inaxes is self.fig.axes[0] and event.button == 1 and event.xdata is not None:
            position = (round(event.ydata), round(event.xdata))
            if self.grid[position] == 0:
                self.add_target(position)
            elif self.grid[position] == TARGET:
                self.clear_target(position)
            # Redrawing the axes renders faster than calling self.display()
            self.fig.axes[0].imshow(self.colors[self.maze])
            masked_smell = np.ma.masked_where(self.grid > 0, self.smell_grid)
            self.fig.axes[0].imshow(masked_smell, cmap=self.cmap_smell, vmin=0, vmax=np.max(masked_smell))

    def save(self, step):
        """Save the display as a PNG file"""
        if "save_dir" in globals():
            fig, ax = plt.subplots()
            ax.set_xticks([])
            ax.set_yticks([])
            ax.axis('off')
            ax.imshow(self.colors[self.maze])
            masked_smell = np.ma.masked_where(self.grid > 0, self.smell_grid)
            ax.imshow(masked_smell, cmap=self.cmap_smell, vmin=0, vmax=np.max(masked_smell))
            plt.scatter(self.position[1], self.position[0], s=self.marker_size, marker=self.marker_map[self.direction], c=self.marker_color)
            ax.text(self.grid.shape[1]-1.5, 0.2, f"{step:>3}", fontsize=12, color='White')
            plt.savefig(f"{save_dir}/{step:03}.png", bbox_inches='tight', pad_inches=0, transparent=True)
            plt.close(fig)
    
    def clear(self, clear):
        """Clear the grid display"""
        if clear:
            self.maze[:, :] = self.grid

    def add_target(self, position):
        """Insert a new target at position (l, c)"""
        self.grid[position] = TARGET
        self.maze[position] = TARGET
        self.smell_map()

    def clear_target(self, position):
        """Remove target at position (l, c)"""
        self.grid[position] = 0
        self.maze[position] = 0    
        self.smell_map()

    def smell_map(self):
        """Construct the smell map using the Dijkstra algorithm"""
        # --- Identify target cells ---
        targets = [(r, c) for r in range(self.grid.shape[0]) for c in range(self.grid.shape[1]) if self.grid[r, c] == TARGET]        
        # --- Compute shortest distances from each cell to the nearest target ---
        self.smell_grid[:, :] = MAX_SMELL
        for target in targets:
            lengths = nx.single_source_dijkstra_path_length(self.G, target)
            for (r, c), dist in lengths.items():
                self.smell_grid[r, c] = min(self.smell_grid[r, c], dist)


# Demonstrate the agent

## Select the `widget` display mode

If your version of Python allows it, you can switch to `widget` display mode by uncommenting the line `%matplotlib widget` in the cell below.
This mode allows adding targets by clicking on the grid. 

You can return to `inLine` mode by uncommenting and runnin the line `%matplotlib inline`. 

In [8]:
# Widget mode supports user interaction with the grid
#%matplotlib widget

# Inline mode does not support user interaction with the grid
# %matplotlib inline

## Initialize the interactions 

The cell below initializes the valances to their default values. 

In [9]:
import ipywidgets as widgets
from IPython.display import display
# Définition des curseurs
slider_trn_left_stable = widgets.IntSlider(value=-5, min=-20, max=10, description='Left stable', continuous_update=False)
slider_trn_left_front = widgets.IntSlider(value=5, min=-20, max=10, description='Left increase', continuous_update=False)
slider_trn_left_decrease = widgets.IntSlider(value=-10, min=-20, max=10, description='Left decrease', continuous_update=False)
vbox1 = widgets.VBox([slider_trn_left_stable, slider_trn_left_front, slider_trn_left_decrease])

slider_fwd_stable = widgets.IntSlider(value=-10, min=-20, max=10, description='Fwd stable', continuous_update=False)
slider_fwd_decrease = widgets.IntSlider(value=-10, min=-20, max=10, description='Fwd decrease', continuous_update=False)
slider_fwd_bump = widgets.IntSlider(value=-10, min=-20, max=10, description='Bump', continuous_update=False)
slider_eat = widgets.IntSlider(value=10, min=-20, max=10, description='Eat', continuous_update=False)
slider_fwd_front = widgets.IntSlider(value=10, min=-20, max=10, description='Fwd increase', continuous_update=False)
vbox2 = widgets.VBox([slider_fwd_stable, slider_fwd_front, slider_fwd_decrease, slider_fwd_bump, slider_eat])

slider_trn_right_stable = widgets.IntSlider(value=-5, min=-20, max=10, description='Right stable', continuous_update=False)
slider_trn_right_front = widgets.IntSlider(value=5, min=-20, max=10, description='Right increase', continuous_update=False)
slider_trn_right_decrease = widgets.IntSlider(value=-10, min=-20, max=10, description='Right decrease', continuous_update=False)
vbox3 = widgets.VBox([slider_trn_right_stable, slider_trn_right_front, slider_trn_right_decrease])

# Affichage des curseurs
display(widgets.HBox([vbox1, vbox2, vbox3]))

You may modify the valances using the cursors above. 

Execute the cell below to initialize the interactions with the valences given by the cursors.

In [10]:
interactions = [
    Interaction(FORWARD, STABLE, slider_fwd_stable.value),
    Interaction(FORWARD, BUMP, slider_fwd_bump.value),
    Interaction(FORWARD, INCREASE, slider_fwd_front.value),
    Interaction(FORWARD, DECREASE, slider_fwd_decrease.value),
    Interaction(FORWARD, EAT, slider_eat.value),
    Interaction(TURN_LEFT, STABLE, slider_trn_left_stable.value),
    Interaction(TURN_LEFT, INCREASE, slider_trn_left_front.value),
    Interaction(TURN_LEFT, DECREASE, slider_trn_left_decrease.value),
    Interaction(TURN_LEFT, EAT, slider_eat.value),
    Interaction(TURN_RIGHT, STABLE, slider_trn_right_stable.value),
    Interaction(TURN_RIGHT, INCREASE, slider_trn_right_front.value),
    Interaction(TURN_RIGHT, DECREASE, slider_trn_right_decrease.value),
    Interaction(TURN_RIGHT, EAT, slider_eat.value),
]

## Initialize the experiment

In [11]:
# Activate printed outputs
trace = True

# Instanciate the environment
#out = Output()
e = Environment([6, 4], UP)

# Instanciate the agent 
a = Agent(interactions)

# Initialize the experiment
step = 0
outcome = 0
# e.display()
# display(out)

## Add a target

You can add more targets at the position `(line, colomn)` of your choice by modifying and reexecuting the cell above.

In [12]:
# Initialize the display
out = Output()
display(out)

# Add a new target at the specified (line, column)
#e.add_target((3, 6))
e.display()

Output()

## Run the agent one step at a time, using `Ctrl+Enter` on the cell below:

In [13]:
print(f"Step: {step}")
step += 1
action = a.action(outcome)
e.display()
e.clear(True)
e.save(step)  # Sauvegarde le fichier image qui servira au gif
outcome = e.outcome(action)
a.proposed_df
# a.decision_df

Step: 0
Action: 0, Prediction: 0, Outcome: 0, Prediction_correct: True, Valence: -10
Intention: 00


,activated,weight,actions,intention,valence,decision,length,pre,proclivity
0,,0,0,00,-10,0,1,,0
1,,0,1,10,-10,1,1,,0
2,,0,2,20,-10,2,1,,0


# Run the agent in a loop

Make sure we are in the `inLine` display mode otherwise the animation does not work.

In [14]:
# Mode inline mode does not allow adding a target by clicking on the figure
%matplotlib inline

Choose the steps and the positions of new targets to insert in `target_steps` et `target_positions`.

In [15]:
# The steps when new targets are inserted
target_steps = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360]
# For each insertion step, give the position of the target (line, column). The keys must correspond to the steps above.
target_positions = {0:(3, 6), 30: (7, 10), 60:(2, 1), 90:(4, 10), 120:(2, 10), 150:(6, 7), 180:(3, 2), 210:(2, 6), 240:(2, 10), 270:(4,2), 300:(5,9), 330:(7,1), 360:(2,10)}

Run the simulation

In [17]:
# Deactivate trace
trace = False

# Instanciate the environment
e = Environment([6, 4], RIGHT)
# Initialize the agent
a = Agent(interactions)
outcome = 0

# Display
out = Output()
e.display()
display(out)
for step in range(400):
    if step in target_steps:
        e.add_target(target_positions[step])
    action = a.action(outcome)
    e.display()
    e.save(step)  # Save a screenshot 
    e.clear(True)
    outcome = e.outcome(action)

Output()

Observe that, after catching a few targets, the agent learns to reach the target efficiently by following a stair-case trajectory until it aligns itself with the target moves straigt to it. 

# Assignment

Modify:
* The valences of the interactions,
* The time and position when new targets appear

and rerun the agent to observe its new behavior. 

## 1. The straightforward agent

Create an agent that turns in place when there is no target and goes directly to the target when it smell one without looking around.

In [421]:
interactions = [
    Interaction(FORWARD, STABLE, -10),
    Interaction(FORWARD, BUMP, -10),
    Interaction(FORWARD, INCREASE, 1),
    Interaction(FORWARD, DECREASE, -1),
    Interaction(FORWARD, EAT, 1),
    Interaction(TURN_LEFT, STABLE, -1),
    Interaction(TURN_LEFT, INCREASE, 1),
    Interaction(TURN_LEFT, DECREASE, -1),
    Interaction(TURN_LEFT, EAT, 1),
    Interaction(TURN_RIGHT, STABLE, -10),
    Interaction(TURN_RIGHT, INCREASE, 1),
    Interaction(TURN_RIGHT, DECREASE, -1),
    Interaction(TURN_RIGHT, EAT, 1),
]
# The steps when new targets are inserted
target_steps = [40, 80, 120, 160, 200, 240, 280, 320, 360]
# For each insertion step, give the position of the target (line, column). The keys must correspond to the steps above.
target_positions = {40:(3, 6), 80: (7, 10), 120:(2, 1), 160:(4, 10), 200:(2, 10), 240:(6, 7), 280:(3, 2), 320:(2, 6), 360:(2, 10), 270:(4,2), 300:(5,9), 330:(7,1), 360:(2,10)}

## 2. The cautious agent

Create an agent that looks around as it goes to a target to check if new targets appeared. 

In [425]:
interactions = [
    Interaction(FORWARD, STABLE, -10),
    Interaction(FORWARD, BUMP, -10),
    Interaction(FORWARD, INCREASE, 10),
    Interaction(FORWARD, DECREASE, -10),
    Interaction(FORWARD, EAT, 10),
    Interaction(TURN_LEFT, STABLE, -5),
    Interaction(TURN_LEFT, INCREASE, 5),
    Interaction(TURN_LEFT, DECREASE, -10),
    Interaction(TURN_LEFT, EAT, 10),
    Interaction(TURN_RIGHT, STABLE, -5),
    Interaction(TURN_RIGHT, INCREASE, 5),
    Interaction(TURN_RIGHT, DECREASE, -10),
    Interaction(TURN_RIGHT, EAT, 10),
]
# The steps when new targets are inserted
target_steps = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360]
# For each insertion step, give the position of the target (line, column). The keys must correspond to the steps above.
target_positions = {0:(3, 6), 30: (7, 10), 60:(2, 1), 90:(4, 10), 120:(2, 10), 150:(6, 7), 180:(3, 2), 210:(2, 6), 240:(2, 10), 270:(4,2), 300:(5,9), 330:(7,1), 360:(2,10)}

# Create a gif animation

Define the directory to save the images. 
When `save_dir` is defined, rerun the agent in a loop. It will save screenshots in the save sub-directeory. 

In [417]:
# The sub-directory to save the images. Use "." to save the images in the same directory as this notebook
save_dir = "sav"  # "."

Generate the gif file

In [420]:
import imageio.v2 as imageio
import os

img_dir = f"./{save_dir}"
# Needs to be sorted on some PCs.
all_files = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith('.png')])
images = [imageio.imread(f) for f in all_files]
imageio.mimsave("movie.gif", images, fps=5)